# Estimadores Sesgados e Insesgados: Simulaciones de Convergencia

En esta notebook exploraremos:

1. Concepto de sesgo de un estimador
2. Ejemplos clásicos de estimadores insesgados y sesgados
3. Simulaciones Monte Carlo para visualizar convergencia
4. Comparación entre sesgo y error cuadrático medio (ECM)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

## 1. Definiciones

**Sesgo de un estimador**: Sea $\hat{\theta}_n$ un estimador del parámetro $\theta$. El sesgo se define como:

$$\text{Sesgo}(\hat{\theta}_n) = E[\hat{\theta}_n] - \theta$$

- **Insesgado**: $E[\hat{\theta}_n] = \theta$ para todo $n$
- **Asintóticamente insesgado**: $\lim_{n \to \infty} E[\hat{\theta}_n] = \theta$

## 2. Ejemplo 1: Media Muestral (Estimador Insesgado)

La media muestral $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$ es un estimador **insesgado** de $\mu = E[X]$.

In [ ]:
# Parámetros verdaderos
mu_verdadero = 5
sigma_verdadero = 2

# Tamaños de muestra a evaluar
tamaños_muestra = np.array([5, 10, 20, 50, 100, 200, 500, 1000, 2000])
n_simulaciones = 10000

# Almacenar resultados
medias_estimadas = np.zeros((len(tamaños_muestra), n_simulaciones))

# Simulación Monte Carlo
for i, n in enumerate(tamaños_muestra):
    for j in range(n_simulaciones):
        muestra = np.random.normal(mu_verdadero, sigma_verdadero, n)
        medias_estimadas[i, j] = np.mean(muestra)

# Calcular sesgo empírico y varianza
sesgo_media = medias_estimadas.mean(axis=1) - mu_verdadero
varianza_media = medias_estimadas.var(axis=1)
ecm_media = ((medias_estimadas - mu_verdadero)**2).mean(axis=1)

In [ ]:
# Visualización: Convergencia de la media muestral
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Distribuciones para diferentes n
ax = axes[0, 0]
indices_seleccionados = [0, 3, 5, 8]  # n = 5, 50, 200, 2000
for idx in indices_seleccionados:
    ax.hist(medias_estimadas[idx, :], bins=50, alpha=0.5, 
            label=f'n = {tamaños_muestra[idx]}', density=True)
ax.axvline(mu_verdadero, color='red', linestyle='--', linewidth=2, label='μ verdadero')
ax.set_xlabel('Valor estimado')
ax.set_ylabel('Densidad')
ax.set_title('Distribución muestral de $\\bar{X}_n$ para diferentes n')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 2: Sesgo vs n
ax = axes[0, 1]
ax.plot(tamaños_muestra, sesgo_media, 'o-', linewidth=2, markersize=8)
ax.axhline(0, color='red', linestyle='--', linewidth=2, label='Sesgo = 0')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Sesgo empírico')
ax.set_title('Sesgo de $\\bar{X}_n$ (Estimador INSESGADO)')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 3: Varianza vs n
ax = axes[1, 0]
varianza_teorica = sigma_verdadero**2 / tamaños_muestra
ax.plot(tamaños_muestra, varianza_media, 'o-', linewidth=2, markersize=8, label='Varianza empírica')
ax.plot(tamaños_muestra, varianza_teorica, 's--', linewidth=2, markersize=6, label='Varianza teórica: σ²/n')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Varianza')
ax.set_title('Varianza de $\\bar{X}_n$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 4: ECM vs n
ax = axes[1, 1]
ax.plot(tamaños_muestra, ecm_media, 'o-', linewidth=2, markersize=8, color='purple')
ax.plot(tamaños_muestra, varianza_teorica, 's--', linewidth=2, markersize=6, 
        label='ECM = Varianza (sesgo = 0)', color='orange')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('ECM')
ax.set_title('Error Cuadrático Medio de $\\bar{X}_n$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== MEDIA MUESTRAL (INSESGADO) ===")
print(f"Parámetro verdadero μ = {mu_verdadero}")
print(f"\nSesgo empírico para n = {tamaños_muestra[-1]}: {sesgo_media[-1]:.6f}")
print(f"Varianza empírica: {varianza_media[-1]:.6f}")
print(f"Varianza teórica (σ²/n): {varianza_teorica[-1]:.6f}")

## 3. Ejemplo 2: Varianza Muestral

Comparación entre dos estimadores de la varianza:

- **Insesgado**: $S^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar{X})^2$
- **Sesgado (MLE)**: $\hat{\sigma}^2 = \frac{1}{n}\sum_{i=1}^n (X_i - \bar{X})^2$

El estimador MLE tiene sesgo: $E[\hat{\sigma}^2] = \frac{n-1}{n}\sigma^2$

In [ ]:
# Simulación para varianza
var_insesgada = np.zeros((len(tamaños_muestra), n_simulaciones))
var_sesgada = np.zeros((len(tamaños_muestra), n_simulaciones))

for i, n in enumerate(tamaños_muestra):
    for j in range(n_simulaciones):
        muestra = np.random.normal(mu_verdadero, sigma_verdadero, n)
        var_insesgada[i, j] = np.var(muestra, ddof=1)  # n-1
        var_sesgada[i, j] = np.var(muestra, ddof=0)    # n

# Calcular sesgo
sesgo_var_insesgada = var_insesgada.mean(axis=1) - sigma_verdadero**2
sesgo_var_sesgada = var_sesgada.mean(axis=1) - sigma_verdadero**2
sesgo_teorico_var_sesgada = -sigma_verdadero**2 / tamaños_muestra

# ECM
ecm_var_insesgada = ((var_insesgada - sigma_verdadero**2)**2).mean(axis=1)
ecm_var_sesgada = ((var_sesgada - sigma_verdadero**2)**2).mean(axis=1)

In [ ]:
# Visualización: Comparación de estimadores de varianza
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Distribuciones para n = 50
ax = axes[0, 0]
idx = 3  # n = 50
ax.hist(var_insesgada[idx, :], bins=50, alpha=0.6, label='S² (insesgado)', density=True, color='blue')
ax.hist(var_sesgada[idx, :], bins=50, alpha=0.6, label='σ̂² (sesgado)', density=True, color='red')
ax.axvline(sigma_verdadero**2, color='green', linestyle='--', linewidth=2, label='σ² verdadero')
ax.axvline(var_insesgada[idx, :].mean(), color='blue', linestyle=':', linewidth=2, alpha=0.7)
ax.axvline(var_sesgada[idx, :].mean(), color='red', linestyle=':', linewidth=2, alpha=0.7)
ax.set_xlabel('Valor estimado')
ax.set_ylabel('Densidad')
ax.set_title(f'Distribución muestral de estimadores de varianza (n = {tamaños_muestra[idx]})')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 2: Sesgo vs n
ax = axes[0, 1]
ax.plot(tamaños_muestra, sesgo_var_insesgada, 'o-', linewidth=2, markersize=8, label='S² (insesgado)')
ax.plot(tamaños_muestra, sesgo_var_sesgada, 's-', linewidth=2, markersize=8, label='σ̂² (sesgado)')
ax.plot(tamaños_muestra, sesgo_teorico_var_sesgada, '--', linewidth=2, 
        label='Sesgo teórico: -σ²/n', color='orange')
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Sesgo empírico')
ax.set_title('Sesgo de estimadores de varianza')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 3: Sesgo relativo
ax = axes[1, 0]
sesgo_relativo_insesgada = sesgo_var_insesgada / sigma_verdadero**2 * 100
sesgo_relativo_sesgada = sesgo_var_sesgada / sigma_verdadero**2 * 100
ax.plot(tamaños_muestra, sesgo_relativo_insesgada, 'o-', linewidth=2, markersize=8, label='S² (insesgado)')
ax.plot(tamaños_muestra, sesgo_relativo_sesgada, 's-', linewidth=2, markersize=8, label='σ̂² (sesgado)')
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Sesgo relativo (%)')
ax.set_title('Sesgo relativo: (Sesgo / σ²) × 100')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 4: ECM vs n
ax = axes[1, 1]
ax.plot(tamaños_muestra, ecm_var_insesgada, 'o-', linewidth=2, markersize=8, label='S² (insesgado)')
ax.plot(tamaños_muestra, ecm_var_sesgada, 's-', linewidth=2, markersize=8, label='σ̂² (sesgado)')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('ECM')
ax.set_title('Error Cuadrático Medio de estimadores de varianza')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== ESTIMADORES DE VARIANZA ===")
print(f"Parámetro verdadero σ² = {sigma_verdadero**2}")
print(f"\nPara n = {tamaños_muestra[-1]}:")
print(f"  S² (insesgado) - Sesgo: {sesgo_var_insesgada[-1]:.6f}")
print(f"  σ̂² (sesgado)   - Sesgo: {sesgo_var_sesgada[-1]:.6f}")
print(f"  Sesgo teórico σ̂²: {sesgo_teorico_var_sesgada[-1]:.6f}")
print(f"\n  S² - ECM: {ecm_var_insesgada[-1]:.6f}")
print(f"  σ̂² - ECM: {ecm_var_sesgada[-1]:.6f}")

## 4. Ejemplo 3: Estimador del Máximo (Distribución Uniforme)

Sea $X_1, \ldots, X_n \sim U(0, \theta)$. Consideramos dos estimadores de $\theta$:

1. **Máximo muestral**: $\hat{\theta}_1 = \max(X_1, \ldots, X_n)$ (SESGADO)
   - $E[\hat{\theta}_1] = \frac{n}{n+1}\theta$
   
2. **Máximo corregido**: $\hat{\theta}_2 = \frac{n+1}{n}\max(X_1, \ldots, X_n)$ (INSESGADO)

In [ ]:
# Parámetro verdadero
theta_verdadero = 10

# Simulación
max_sesgado = np.zeros((len(tamaños_muestra), n_simulaciones))
max_insesgado = np.zeros((len(tamaños_muestra), n_simulaciones))

for i, n in enumerate(tamaños_muestra):
    for j in range(n_simulaciones):
        muestra = np.random.uniform(0, theta_verdadero, n)
        max_muestra = np.max(muestra)
        max_sesgado[i, j] = max_muestra
        max_insesgado[i, j] = ((n + 1) / n) * max_muestra

# Calcular sesgo
sesgo_max_sesgado = max_sesgado.mean(axis=1) - theta_verdadero
sesgo_max_insesgado = max_insesgado.mean(axis=1) - theta_verdadero
sesgo_teorico_max = -theta_verdadero / (tamaños_muestra + 1)

# ECM
ecm_max_sesgado = ((max_sesgado - theta_verdadero)**2).mean(axis=1)
ecm_max_insesgado = ((max_insesgado - theta_verdadero)**2).mean(axis=1)

In [ ]:
# Visualización: Estimadores del máximo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Distribuciones
ax = axes[0, 0]
idx = 2  # n = 20
ax.hist(max_sesgado[idx, :], bins=50, alpha=0.6, label='max(X) (sesgado)', density=True, color='red')
ax.hist(max_insesgado[idx, :], bins=50, alpha=0.6, label='(n+1)/n·max(X) (insesgado)', 
        density=True, color='blue')
ax.axvline(theta_verdadero, color='green', linestyle='--', linewidth=2, label='θ verdadero')
ax.set_xlabel('Valor estimado')
ax.set_ylabel('Densidad')
ax.set_title(f'Distribución de estimadores de θ en U(0,θ) (n = {tamaños_muestra[idx]})')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 2: Sesgo vs n
ax = axes[0, 1]
ax.plot(tamaños_muestra, sesgo_max_sesgado, 's-', linewidth=2, markersize=8, label='max(X) (sesgado)')
ax.plot(tamaños_muestra, sesgo_max_insesgado, 'o-', linewidth=2, markersize=8, label='(n+1)/n·max(X)')
ax.plot(tamaños_muestra, sesgo_teorico_max, '--', linewidth=2, 
        label='Sesgo teórico: -θ/(n+1)', color='orange')
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Sesgo')
ax.set_title('Sesgo de estimadores del parámetro θ')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 3: Convergencia del estimador sesgado
ax = axes[1, 0]
esperanza_teorica = tamaños_muestra / (tamaños_muestra + 1) * theta_verdadero
ax.plot(tamaños_muestra, max_sesgado.mean(axis=1), 'o-', linewidth=2, markersize=8, 
        label='E[max(X)] empírico')
ax.plot(tamaños_muestra, esperanza_teorica, 's--', linewidth=2, markersize=6, 
        label='E[max(X)] teórico: n·θ/(n+1)')
ax.axhline(theta_verdadero, color='green', linestyle='--', linewidth=2, label='θ verdadero')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Esperanza')
ax.set_title('Convergencia asintótica de E[max(X)] → θ')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 4: ECM comparado
ax = axes[1, 1]
ax.plot(tamaños_muestra, ecm_max_sesgado, 's-', linewidth=2, markersize=8, label='max(X) (sesgado)')
ax.plot(tamaños_muestra, ecm_max_insesgado, 'o-', linewidth=2, markersize=8, 
        label='(n+1)/n·max(X) (insesgado)')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('ECM')
ax.set_title('ECM: Sesgo vs Varianza Trade-off')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== ESTIMADORES DEL MÁXIMO U(0,θ) ===")
print(f"Parámetro verdadero θ = {theta_verdadero}")
print(f"\nPara n = {tamaños_muestra[-1]}:")
print(f"  max(X) - E[max(X)] empírico: {max_sesgado[-1, :].mean():.6f}")
print(f"  max(X) - E[max(X)] teórico: {esperanza_teorica[-1]:.6f}")
print(f"  max(X) - Sesgo: {sesgo_max_sesgado[-1]:.6f}")
print(f"  (n+1)/n·max(X) - Sesgo: {sesgo_max_insesgado[-1]:.6f}")
print(f"\n  max(X) - ECM: {ecm_max_sesgado[-1]:.6f}")
print(f"  (n+1)/n·max(X) - ECM: {ecm_max_insesgado[-1]:.6f}")
print(f"\n  Nota: El estimador SESGADO tiene menor ECM!")

## 5. Ejemplo 4: Estimador de la Desviación Estándar

La raíz cuadrada de la varianza muestral $S = \sqrt{S^2}$ es un estimador **sesgado** de $\sigma$, aunque $S^2$ sea insesgado.

Por la desigualdad de Jensen: $E[\sqrt{S^2}] < \sqrt{E[S^2]} = \sigma$

In [ ]:
# Simulación para desviación estándar
std_muestra = np.zeros((len(tamaños_muestra), n_simulaciones))

for i, n in enumerate(tamaños_muestra):
    for j in range(n_simulaciones):
        muestra = np.random.normal(mu_verdadero, sigma_verdadero, n)
        std_muestra[i, j] = np.std(muestra, ddof=1)

# Factor de corrección para hacer insesgado (distribución normal)
# c_n para Normal: E[S] = σ · c_n, donde c_n → 1 cuando n → ∞
from scipy.special import gamma
c_n = np.sqrt(2 / (tamaños_muestra - 1)) * gamma(tamaños_muestra / 2) / gamma((tamaños_muestra - 1) / 2)
std_corregida = std_muestra / c_n[:, np.newaxis]

sesgo_std = std_muestra.mean(axis=1) - sigma_verdadero
sesgo_std_corregida = std_corregida.mean(axis=1) - sigma_verdadero

In [ ]:
# Visualización: Desviación estándar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Sesgo de S
ax = axes[0]
ax.plot(tamaños_muestra, sesgo_std, 'o-', linewidth=2, markersize=8, label='S (sesgado)')
ax.plot(tamaños_muestra, sesgo_std_corregida, 's-', linewidth=2, markersize=8, 
        label='S/c_n (insesgado)')
ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Sesgo')
ax.set_title('Sesgo del estimador de σ')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Gráfico 2: Factor de corrección c_n
ax = axes[1]
ax.plot(tamaños_muestra, c_n, 'o-', linewidth=2, markersize=8, color='purple')
ax.axhline(1, color='red', linestyle='--', linewidth=2, label='c_n → 1 cuando n → ∞')
ax.set_xlabel('Tamaño de muestra (n)')
ax.set_ylabel('Factor c_n')
ax.set_title('Factor de corrección para insesgamiento')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== ESTIMADOR DE σ ===")
print(f"Parámetro verdadero σ = {sigma_verdadero}")
print(f"\nPara n = {tamaños_muestra[-1]}:")
print(f"  S - E[S] empírico: {std_muestra[-1, :].mean():.6f}")
print(f"  S - Sesgo: {sesgo_std[-1]:.6f}")
print(f"  S/c_n - Sesgo: {sesgo_std_corregida[-1]:.6f}")
print(f"  Factor c_n = {c_n[-1]:.6f}")

## 6. Tabla Resumen Comparativa

In [ ]:
# Crear tabla resumen para n grande
idx_grande = -1  # n = 2000

resumen = pd.DataFrame({
    'Estimador': [
        'Media: X̄',
        'Varianza: S²',
        'Varianza: σ̂²',
        'Desv. Est.: S',
        'Desv. Est.: S/c_n',
        'Máximo: max(X)',
        'Máximo: (n+1)/n·max(X)'
    ],
    'Parámetro': ['μ', 'σ²', 'σ²', 'σ', 'σ', 'θ', 'θ'],
    'Tipo': ['Insesgado', 'Insesgado', 'Sesgado', 'Sesgado', 'Insesgado', 'Sesgado', 'Insesgado'],
    'Sesgo Empírico': [
        f"{sesgo_media[idx_grande]:.6f}",
        f"{sesgo_var_insesgada[idx_grande]:.6f}",
        f"{sesgo_var_sesgada[idx_grande]:.6f}",
        f"{sesgo_std[idx_grande]:.6f}",
        f"{sesgo_std_corregida[idx_grande]:.6f}",
        f"{sesgo_max_sesgado[idx_grande]:.6f}",
        f"{sesgo_max_insesgado[idx_grande]:.6f}"
    ],
    'ECM': [
        f"{ecm_media[idx_grande]:.6f}",
        f"{ecm_var_insesgada[idx_grande]:.6f}",
        f"{ecm_var_sesgada[idx_grande]:.6f}",
        'N/A',
        'N/A',
        f"{ecm_max_sesgado[idx_grande]:.6f}",
        f"{ecm_max_insesgado[idx_grande]:.6f}"
    ]
})

print(f"\n{'='*80}")
print(f"TABLA RESUMEN COMPARATIVA (n = {tamaños_muestra[idx_grande]})")
print(f"{'='*80}")
print(resumen.to_string(index=False))
print(f"{'='*80}")

## 7. Conclusiones

### Observaciones clave:

1. **Sesgo vs Consistencia**:
   - Un estimador sesgado puede ser **asintóticamente insesgado** (ej: $\hat{\sigma}^2$, $\max(X)$)
   - El sesgo tiende a 0 cuando $n \to \infty$ en muchos casos prácticos

2. **Sesgo vs ECM**:
   - Un estimador sesgado puede tener **menor ECM** que uno insesgado
   - Trade-off sesgo-varianza: $\text{ECM} = \text{Var} + \text{Sesgo}^2$
   - Ejemplo: $\max(X)$ tiene menor ECM que $(n+1)/n \cdot \max(X)$ para $n$ pequeño

3. **Estimadores Insesgados**:
   - Media muestral $\bar{X}$ → siempre insesgada
   - Varianza muestral $S^2$ → insesgada
   - Desviación estándar $S$ → sesgada (no linealidad)

4. **Convergencia**:
   - Ley de los Grandes Números garantiza convergencia en probabilidad
   - La varianza decrece típicamente como $O(1/n)$
   - El sesgo puede decrecer más rápido que la varianza

### Recomendación práctica:

- Para $n$ grande: el sesgo suele ser irrelevante (convergencia asintótica)
- Para $n$ pequeño: considerar el ECM total, no solo el sesgo
- En inferencia: preferir estimadores insesgados cuando sea posible
- En predicción: el ECM es más importante que el sesgo per se